Recommend only using Pyvis to create graphs for single CDRs or heavily filtered data from across multiple CDRs.

In [269]:
from pyvis.network import Network
import pandas as pd

TARGET_PATH = '../outputs_sample_1/notional_call_detail_records/CROSS_BORDER_DRIVER001_CDR.csv'
OUTPUT_NAME = 'pyvis_graph_for_single_cdr.html'

In [270]:
df = pd.read_csv(TARGET_PATH)
print(df.shape)
df.head()

(4815, 15)


,RecordID,SubscriberID,SubscriberRole,EventTimestamp,Direction,EventType,CallingNumber,CalledNumber,DurationSec,SMSLength,CellTowerID,TowerName,Latitude,Longitude,Country
0,K4IMM3G03O,CROSS_BORDER_DRIVER001,CROSS_BORDER_DRIVER,2026-04-02 10:45:53,MO,SMS,529296149297,524008645585,NaN,148.0,SN001,Hermosillo Centro,29.0892,-110.9613,MX
1,17ZN4WFEL0,CROSS_BORDER_DRIVER001,CROSS_BORDER_DRIVER,2026-05-19 05:47:33,MO,VOICE,529296149297,524008645585,180.0,NaN,SN001,Hermosillo Centro,29.0892,-110.9613,MX
2,FT2TQFSSSW,CROSS_BORDER_DRIVER001,CROSS_BORDER_DRIVER,2026-02-27 00:57:14,MO,SMS,529296149297,524008645585,NaN,43.0,SN002,Hermosillo Norte,29.1261,-110.9554,MX
3,XZMGS47JJX,CROSS_BORDER_DRIVER001,CROSS_BORDER_DRIVER,2026-04-03 12:34:42,MO,VOICE,529296149297,524008645585,509.0,NaN,US201,San Diego East,32.7157,-117.1611,US
4,VRUSKUL6F7,CROSS_BORDER_DRIVER001,CROSS_BORDER_DRIVER,2026-04-16 19:05:31,MO,VOICE,529296149297,524008645585,258.0,NaN,US204,LA South Bay,33.9019,-118.4173,US


In [271]:
# Groupby on calling, called, and event type to get counts for each event
grouped_events = df.groupby(['CallingNumber', 'CalledNumber', 'EventType'])['RecordID'].count().reset_index()
grouped_events['EventTypeCount'] = grouped_events.apply(lambda x: f"{x['RecordID']} {x['EventType']}", axis=1)

agg_events = grouped_events.groupby(['CallingNumber', 'CalledNumber']).agg({
        'EventType': ', '.join,
        'RecordID': 'sum',
        'EventTypeCount': ', '.join
    }).reset_index()

agg_events.rename(mapper={
    'RecordID': 'TotalEventCount',
    'EventTypeCount': 'EventCountByType'
}, axis=1, inplace=True)

agg_events

,CallingNumber,CalledNumber,EventType,TotalEventCount,EventCountByType
0,12137093109,529296149297,"SMS, VOICE",51,"21 SMS, 30 VOICE"
1,12137211469,529296149297,"SMS, VOICE",118,"51 SMS, 67 VOICE"
2,12138104549,529296149297,"SMS, VOICE",56,"25 SMS, 31 VOICE"
3,13102858592,529296149297,"SMS, VOICE",70,"25 SMS, 45 VOICE"
4,13104116844,529296149297,"SMS, VOICE",120,"47 SMS, 73 VOICE"
...,...,...,...,...,...
57,529296149297,525474039613,"SMS, VOICE",58,"16 SMS, 42 VOICE"
58,529296149297,526013978225,"SMS, VOICE",85,"26 SMS, 59 VOICE"
59,529296149297,528371261458,"SMS, VOICE",148,"59 SMS, 89 VOICE"
60,529296149297,528759319312,"SMS, VOICE",13,"5 SMS, 8 VOICE"


In [272]:
net = Network(
    width='100%',
    height='750px',
    directed=True,
    bgcolor='white',
    font_color='black'
    )

In [273]:
all_nodes = list(set(
    list(agg_events.CallingNumber.unique())
    + list(agg_events.CalledNumber.unique())
))
all_nodes = [int(i) for i in all_nodes]
print(len(all_nodes))
all_nodes[0:3]

32


[16029367428, 16028177419, 528759319312]

In [274]:
all_edges = list(agg_events.itertuples(index=False, name=None))
all_edges[0:3]

[(12137093109, 529296149297, 'SMS, VOICE', 51, '21 SMS, 30 VOICE'),
 (12137211469, 529296149297, 'SMS, VOICE', 118, '51 SMS, 67 VOICE'),
 (12138104549, 529296149297, 'SMS, VOICE', 56, '25 SMS, 31 VOICE')]

In [ ]:
# Or add individually (more control over styling)
for n in all_nodes:
    net.add_node(
        n_id=n,
        label=str(n), 
        title=str(n),
        shape='image', 
        image='phone_circle.png',
        size=10,
        physics=False
    )

print(len(net.get_nodes()))

32


In [276]:
for e in all_edges:
    net.add_edge(
        source=e[0],
        to=e[1],
        # value=e[3],
        label=str(e[3]),
        title=e[4],
        color={
            'color': 'grey',
            'highlight': 'red',
            'inherit': False,
        },
        physics=False
    )

print(len(net.get_edges()))

62


In [ ]:
# net.toggle_physics(False)
net.set_edge_smooth('straightCross')
net.save_graph(OUTPUT_NAME)
print('done')

done
